<a href="https://colab.research.google.com/github/UrvashiJ/Major-Depressive-Disorder-DSM-5-Heterogeneity-Visualization/blob/main/VisualizationofHeterogenitiyinMDD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Major Depressive Disorder (DSM-5) Heterogeneity Visualization
# ADVANCED INTERACTIVE VERSION (Plotly-based)

# Install required libraries (run once)
# !pip install pandas plotly networkx

import itertools
import pandas as pd
import networkx as nx
import plotly.express as px
import plotly.graph_objects as go
from collections import Counter

# -----------------------------
# 1. Define symptoms
# -----------------------------

symptoms = [
    "Depressed Mood",
    "Anhedonia",
    "Weight/Appetite Change",
    "Sleep Disturbance",
    "Psychomotor Change",
    "Fatigue",
    "Worthlessness/Guilt",
    "Concentration Problems",
    "Suicidal Ideation"
]

core_symptoms = {"Depressed Mood", "Anhedonia"}

# -----------------------------
# 2. Generate valid combinations
# -----------------------------

valid_sets = []

for k in range(5, 10):
    for combo in itertools.combinations(symptoms, k):
        if core_symptoms.intersection(combo):
            valid_sets.append(combo)

print(f"Total valid DSM-5 symptom profiles: {len(valid_sets)}")

# -----------------------------
# 3. Convert to DataFrame
# -----------------------------

data = []
for combo in valid_sets:
    row = {symptom: int(symptom in combo) for symptom in symptoms}
    data.append(row)

_df = pd.DataFrame(data)

# -----------------------------
# 4. INTERACTIVE BAR CHART
# -----------------------------

freq = _df.sum().sort_values(ascending=False)

fig = px.bar(
    x=freq.index,
    y=freq.values,
    color=["Core" if s in core_symptoms else "Other" for s in freq.index],
    title="Interactive Symptom Frequency",
    labels={'x':'Symptom','y':'Count'}
)

fig.show()

# -----------------------------
# 5. INTERACTIVE HEATMAP
# -----------------------------

co_matrix = pd.DataFrame(0, index=symptoms, columns=symptoms)

for combo in valid_sets:
    for a, b in itertools.combinations(combo, 2):
        co_matrix.loc[a, b] += 1
        co_matrix.loc[b, a] += 1

fig = px.imshow(
    co_matrix,
    text_auto=True,
    title="Interactive Symptom Co-occurrence Heatmap",
    color_continuous_scale="RdBu"
)

fig.show()

# -----------------------------
# 6. INTERACTIVE NETWORK GRAPH
# -----------------------------

G = nx.Graph()
G.add_nodes_from(symptoms)

co_occurrence = Counter()

for combo in valid_sets:
    for a, b in itertools.combinations(combo, 2):
        co_occurrence[(a, b)] += 1

for (a, b), weight in co_occurrence.items():
    G.add_edge(a, b, weight=weight)

pos = nx.spring_layout(G, seed=42)

edge_x = []
edge_y = []

for edge in G.edges():
    x0, y0 = pos[edge[0]]
    x1, y1 = pos[edge[1]]
    edge_x.extend([x0, x1, None])
    edge_y.extend([y0, y1, None])

edge_trace = go.Scatter(
    x=edge_x, y=edge_y,
    line=dict(width=1, color='#888'),
    hoverinfo='none',
    mode='lines')

node_x = []
node_y = []
node_text = []
node_color = []

for node in G.nodes():
    x, y = pos[node]
    node_x.append(x)
    node_y.append(y)
    node_text.append(node)
    node_color.append('red' if node in core_symptoms else 'blue')

node_trace = go.Scatter(
    x=node_x, y=node_y,
    mode='markers+text',
    text=node_text,
    textposition="top center",
    hoverinfo='text',
    marker=dict(
        size=20,
        color=node_color,
        line_width=2))

fig = go.Figure(data=[edge_trace, node_trace],
             layout=go.Layout(
                title='Interactive Symptom Network',
                showlegend=False,
                hovermode='closest'))

fig.show()

# -----------------------------
# 7. FILTER BY SYMPTOM (POWERFUL)
# -----------------------------

# Example: filter profiles containing Suicidal Ideation
filtered = _df[_df["Suicidal Ideation"] == 1]

print(f"Profiles with Suicidal Ideation: {len(filtered)}")

# -----------------------------
# 8. SAVE DATA
# -----------------------------

_df.to_csv("mdd_symptom_profiles.csv", index=False)

print("Dataset saved.")

# -----------------------------
# END
# -----------------------------

Total valid DSM-5 symptom profiles: 227


Profiles with Suicidal Ideation: 141
Dataset saved.
